In [1]:
#Imports
import pandas as pd
import glob
import os

## Importing Data

To start, we are going to be combining all the neuron site data into one data frame.

The original dataset is split across 193 separate CSV files, and each file corresponds to a single neuron that was recorded independently from the AM face patch. Inside each file, the rows represent repeated trials where the monkey viewed one of the face images, and the columns represent time in milliseconds after the image appeared. The values are binary, so a `1` means the neuron fired at that specific millisecond and a `0` means it did not. The reason the dataset is separated like this is simply due to how single neurons are recorded in electrophysiology. Each neuron is isolated at a different time, saved into its own file, and then the electrode is moved to find the next neuron.

Even though the neurons were collected separately, they all saw the exact same stimuli in the same order. This means that trial 1 in every file corresponds to the same identity and the same head orientation, trial 2 corresponds to the next identity and orientation, and so on. Because of that, it makes sense to combine the neurons into one dataset. Each neuron only carries a small amount of information on its own, so combining them allows us to build a population level representation that is much more informative and much more suitable for decoding identity or head orientation.

In [2]:
search_path = "data/*.csv" 
all_files = glob.glob(search_path, recursive=True)
processed_dfs = []
raster_files = [f for f in all_files if "raster_data" in f.lower()]

print(f"Found {len(raster_files)} files.")
processed_dfs = []
for filename in raster_files:
    df = pd.read_csv(filename) 
    site_id = filename.split('site')[1].replace('.csv', '')
    df.insert(0, 'site', site_id)
    processed_dfs.append(df)

data = pd.concat(processed_dfs, axis=0, ignore_index=True)
data.to_csv("data/data.csv", index=False)

Found 193 files.


In [3]:
processed_dfs = []
for filename in raster_files:
    df = pd.read_csv(filename) 
    site_id = filename.split('site')[1].replace('.csv', '')
    df.insert(0, 'site', site_id)
    processed_dfs.append(df)

data = pd.concat(processed_dfs, axis=0, ignore_index=True)
data.to_csv("data/data.csv", index=False)

Right now, the data is loaded vertically, meaning that all of the neurons have been stacked on top of each other in a single dataframe. In this format, each row corresponds to one trial from one neuron, and the site column tells us which neuron produced that row. However, this vertical format isn't super useful for decoding. Since each neuron saw the same sequence of trials, we can take advantage of the fact that trial numbers line up across all sites. This makes it possible to reorganize the dataset horizontally, where each trial becomes a single row and all neurons appear as separate columns. This horizontal layout is much more informative because it represents the population activity for each stimulus presentation, rather than keeping neurons separated. 

In [ ]:
# 3. Load labels (adjust path / column names if needed)
labels = pd.read_csv("data/labels.csv")
identity = labels["identity"]
orientation = labels["orientation"]

# 4. Reshape vertical data into horizontal pseudo population
df = data.copy()

#only columns except 'site' are timepoints
value_cols = [c for c in df.columns if c != "site"]
df["trial"] = df.groupby("site").cumcount()
df_indexed = df.set_index(["trial", "site"])[value_cols]

# Unstack 'site' so that each neuron becomes a set of columns
# Columns become a MultiIndex: (timepoint, site)
horizontal = df_indexed.unstack("site")

# Flatten the MultiIndex columns into simple names like 'site013_t0', 'site013_t1', etc.
horizontal.columns = [f"site{col[1]}_{col[0]}" for col in horizontal.columns]

# Reset the trial index to a plain column index
horizontal.reset_index(drop=True, inplace=True)

# 5. Add labels at the end
horizontal["identity"] = identity
horizontal["orientation"] = orientation

# 6. Save horizontal pseudo population
horizontal.to_csv("data/horizontal_population.csv", index=False)
print("Saved horizontal population data to data/horizontal_population.csv")